# Linear System Test of the Trajectory Manifold Learn Controller

In [1]:
import torch
import numpy as np

device = torch.device("cuda")
W = None
x_dim = 4
u_dim = 1
H = 10
spectral_radius = 0.9
seed=234

x0 = np.zeros(x_dim)
x_ref = np.zeros(x_dim)
u_ref = np.zeros(u_dim)

Q = torch.eye(x_dim, device=device)
R = torch.eye(u_dim, device=device)
x_ref_t = torch.tensor(x_ref, dtype=torch.float32, device=device)
u_ref_t = torch.tensor(u_ref, dtype=torch.float32, device=device)

w_dim = (H + 1) * x_dim + H * u_dim



## Sample Controllable System

In [2]:
from src.linear_system import sample_controllable_linear_system, is_controllable

A, B = sample_controllable_linear_system(
        x_dim,
        u_dim,
        spectral_radius=spectral_radius,
        seed=seed,
)
rank, controllable = is_controllable(A, B)
print(f"  controllability rank={rank}, full={controllable}")

  controllability rank=4, full=True


## Generate System Trajectories

In [1]:
import importlib
import src.linear_system
importlib.reload(src.linear_system)

<module 'src.linear_system' from 'c:\\Users\\bayer\\PycharmProjects\\koopman\\src\\linear_system.py'>

In [3]:
num_steps = 100
n_repeats = 100
process_noise_std = 0.0

In [4]:
from src.linear_system import generate_linear_trajectory_data
from src.manifold_control import build_trajectory_training_matrix

_, X_all, U_all = generate_linear_trajectory_data(
    A,
    B,
    num_steps=num_steps,
    n_repeats=n_repeats,
    process_noise_std=process_noise_std,
    seed=seed,
)

W = build_trajectory_training_matrix(
    X_all,
    U_all,
    horizon=H,
    device=device,
    dtype=torch.float32,
)
w_dim = W.shape[1]
print(f"  training matrix shape={tuple(W.shape)}")

  training matrix shape=(9100, 54)


# Train Manifold Decoder

In [5]:
from pathlib import Path

alpha_dim = 16
hidden_dims = [64, 64]
epochs = 100
max_iter = 1000
train_lr = 1e-3

print_every = 10
checkpoint = Path("saves") / "saved_models" / "behavior_decoder_linear.pt"

In [ ]:
from src.manifold_control import train_decoder

decoder = train_decoder(
    W,
    x_dim=x_dim,
    u_dim=u_dim,
    horizon=H,
    alpha_dim=alpha_dim,
    hidden_dims=hidden_dims,
    epochs=epochs,
    max_iter=max_iter,
    lr=train_lr,
    print_every=print_every,
    checkpoint=checkpoint,
    device=device,
)
decoder.eval()

/home/ab126/programs/micromamba/envs/control311/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


# Solve the Control problem

In [ ]:
x_ref = np.zeros(x_dim)
u_ref = np.zeros(u_dim)
lambda_theta = 1.0
lambda_curvature = 1.0
solve_lr = 1e-3
u_max = 100.0

## Single Step Control

In [ ]:
from src.manifold_control import BehaviorManifoldControlSolver


solver = BehaviorManifoldControlSolver(
        decoder=decoder,
        x_dim=x_dim,
        u_dim=u_dim,
        horizon=H,
        Q=Q,
        R=R,
        x_ref=x_ref,
        u_ref=u_ref,
        lambda_theta=lambda_theta,
        lambda_curvature=lambda_curvature,
        lr=solve_lr,
        max_iter=max_iter,
        u_bounds=(-u_max, u_max),
        curvature_mode="local",
        device=device,
    )

x_init = torch.zeros(H + 1, x_dim, device=device)
x_init[0] = torch.tensor(x0, dtype=torch.float32, device=device)
u_init = torch.zeros(H, u_dim, device=device)
alpha_init = torch.zeros(alpha_dim, device=device)

solution = solver.solve(
    x_init=x_init,
    u_init=u_init,
    alpha_init=alpha_init,
    freeze={"theta": True, "x": False, "u": False, "alpha": False},
)

u_plan = solution.u.detach().cpu().numpy()
print(f"  loss_dict={solution.loss_dict}")
print(f"  first control={u_plan[0]}")

## Closed Loop Control